# Phase 6B — Phases 1+2+3 verification (post-D9 root-cause fixes)

**Phase 1** — `core/brain_graph.py` `predicted_angles_next = jc_out`
(setpoint semantics; previously `q + 0.1·jc` velocity-proxy mismatched
with the body's MuJoCo position actuator).

**Phase 2** — `core/vta.py` `auto_decay`/`baseline_decay` raised to
`n_substeps` power inside `vta_compute_rpe`. Decays were stored as
per-substep values but the function is called once per cognitive cycle
(= 20 substeps), so effective τ was 20× longer than designed. Fix
restores τ_baseline=30 s, τ_auto=10 s.

**Phase 3** — same correction in `core/world_model.py` `wm_update`
(`pe_short`, `pe_long`); restores τ_short=100 ms, τ_long=10 s.

All three fixes are unit-correctness, not tuning.

In [ ]:
import os, sys, subprocess
if not os.path.isdir('/content/Neuro_MVP_6'):
    subprocess.check_call(['git', 'clone', '-q',
        'https://github.com/MateuszSieczka/Neuro_MVP.git', '/content/Neuro_MVP_6'])
%cd /content/Neuro_MVP_6
!pip install -q mujoco mujoco-mjx 'jax[cuda12]' equinox 'numpy<2'
if '/content/Neuro_MVP_6' not in sys.path:
    sys.path.insert(0, '/content/Neuro_MVP_6')
import jax
print('jax', jax.__version__, 'devices:', jax.devices())

/content/Neuro_MVP_6
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.5/42.5 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 78.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 112.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.7/177.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 105.9 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.8/2.8 MB 104.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 116.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.1/123.1 MB 7.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 81.2/81.2 MB 186.7 MB/s eta 0:00:0100:01
ERROR: Operation cancelled by user


In [ ]:
# V1 — Source-level confirmation of the three patches.
import inspect
from core import brain_graph, vta, world_model
src_bg  = inspect.getsource(brain_graph.action_brain_cognitive_step)
src_vta = inspect.getsource(vta.vta_compute_rpe)
src_wm  = inspect.getsource(world_model.wm_update)
checks = {
    'P1: predicted_angles_next = jc_out': 'predicted_angles_next = jnp.clip(\n            jc_out,' in src_bg,
    'P1: old velocity-proxy gone':         '+ jnp.asarray(0.1, DTYPE) * jc_out' not in src_bg,
    'P2: auto_decay_cycle present':        'auto_decay_cycle' in src_vta,
    'P2: baseline_decay_cycle present':    'baseline_decay_cycle' in src_vta,
    'P3: pe_short_decay_cycle present':    'pe_short_decay_cycle' in src_wm,
    'P3: pe_long_decay_cycle present':     'pe_long_decay_cycle' in src_wm,
}
for k, v in checks.items():
    print(('PASS' if v else 'FAIL'), '—', k)
assert all(checks.values())

In [ ]:
# V2 — Numerical effect: per-cycle decay equals exp(-n_sub*dt/τ).
import jax.numpy as jnp
from core.backend import BackendContext
ctx = BackendContext()
tau = 30_000.0
n_sub = 20
per_step = float(ctx.decay(tau))
per_cycle_fix = per_step ** n_sub
expected = float(jnp.exp(-n_sub * ctx.dt / tau))
print(f'per-step decay              = {per_step:.6f}')
print(f'per-cycle decay (fix)       = {per_cycle_fix:.6f}')
print(f'expected exp(-n*dt/τ)       = {expected:.6f}')
assert abs(per_cycle_fix - expected) < 1e-7

In [ ]:
# V3 — Drive 400-cycle babble; record telemetry.
import jax, numpy as np, equinox as eqx, pandas as pd
from core.backend import DTYPE
from embodiment.reacher_env import build_reacher
from embodiment.mjx_run_loop import _one_babble_cycle
key = jax.random.PRNGKey(0)
params, state0, body = build_reacher(key, n_cycles=1)
ctx = BackendContext()

def telem(s):
    return dict(
        jc0=float(s.m1.last_joint_command[0]),
        jc1=float(s.m1.last_joint_command[1]),
        pred_q0=float(s.last_predicted_joint_angles[0]),
        pred_q1=float(s.last_predicted_joint_angles[1]),
        dn_rate_mean=float(jnp.mean(jnp.abs(s.cerebellum.dn_rate))),
        wm_pe_short=float(s.world_model.pe_short_abs),
        wm_pe_long=float(s.world_model.pe_long_abs),
        rpe=float(s.vta.last_rpe),
        V=float(s.vta.stored_v),
        baseline=float(s.vta.reward_baseline),
        auto_rms=float(s.vta.auto_rms),
        W_norm=float(jnp.linalg.norm(s.m1.W)),
    )

carry = (state0, body, body.zero_sensory(), jnp.asarray(0., DTYPE), jnp.asarray(0., DTYPE))
k = jax.random.PRNGKey(1)
logs = []
for t in range(400):
    k, ki = jax.random.split(k)
    carry, _ = _one_babble_cycle(carry, ki, brain_params=params, ctx=ctx)
    if t % 10 == 0:
        logs.append({'t': t, **telem(carry[0])})
df = pd.DataFrame(logs)
print(df.tail(8).round(4))

In [ ]:
# V4 — Phase 1 invariant: pred_q == jc.
diff = np.max(np.abs(df[['pred_q0','pred_q1']].values - df[['jc0','jc1']].values))
print(f'max |pred_q − jc| = {diff:.6e}')
assert diff < 1e-5

In [ ]:
# V5 — Cumulative trajectories.
first = df.iloc[:5].mean(numeric_only=True)
last  = df.iloc[-5:].mean(numeric_only=True)
print(pd.DataFrame({'first': first, 'last': last, 'Δ': last - first}).round(4)
      .loc[['dn_rate_mean','wm_pe_short','wm_pe_long','baseline','auto_rms','W_norm','V','rpe']])

In [ ]:
# V6 — Plot.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 3, figsize=(13, 6))
ax[0,0].plot(df.t, df.W_norm); ax[0,0].set_title('|W|_F')
ax[0,1].plot(df.t, df.wm_pe_short, label='short'); ax[0,1].plot(df.t, df.wm_pe_long, label='long'); ax[0,1].legend(); ax[0,1].set_title('wm |pe|')
ax[0,2].plot(df.t, df.baseline, label='baseline'); ax[0,2].legend(); ax[0,2].set_title('vta baseline')
ax[1,0].plot(df.t, df.V); ax[1,0].set_title('V(s)')
ax[1,1].plot(df.t, df.rpe); ax[1,1].set_title('RPE')
ax[1,2].plot(df.t, df.dn_rate_mean); ax[1,2].set_title('cerebellum |dn|')
for a in ax.flat: a.set_xlabel('cycle')
plt.tight_layout(); plt.show()